# ARTI308 - Machine Learning
# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
print(f'Shape: {df.shape}')
print(f'Rows   : {df.shape[0]}')
print(f'Columns: {df.shape[1]}')

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe().round(2)

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop(columns=['CUST_ID'], inplace=True)
print('CUST_ID dropped. Remaining columns:', df.shape[1])

**Check the missing values in each column.**

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)
print('Missing values filled with column mean.')

**Check the missing values again to make sure they were handled.**

In [ ]:
print('Remaining missing values:')
print(df.isnull().sum().sum(), '← should be 0')

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(18, 14), bins=30, color='steelblue', edgecolor='white')
plt.suptitle('Feature Distributions — Credit Card Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.3, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap — Credit Card Features', fontsize=13)
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.4, s=10, color='steelblue')
plt.xlabel('BALANCE')
plt.ylabel('PURCHASES')
plt.title('Balance vs Purchases')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.4, s=10, color='darkorange')
plt.xlabel('BALANCE')
plt.ylabel('CASH_ADVANCE')
plt.title('Balance vs Cash Advance')
plt.tight_layout()
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print(f'X_scaled shape : {X_scaled.shape}')
print(f'Mean (first col): {X_scaled[:, 0].mean():.4f}  (should be ~0)')
print(f'Std  (first col): {X_scaled[:, 0].std():.4f}  (should be ~1)')

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []
K_range = range(1, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

print('Inertia values computed for K = 1 to 10.')

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(K_range, inertia_values, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Choosing K')
plt.xticks(K_range)
plt.tight_layout()
plt.show()

**Output Interpretation**

The elbow curve shows inertia decreasing as K increases. The point where the curve starts to flatten (the "elbow") suggests a good value for K. In this dataset, the curve starts to flatten around **K = 4**, meaning adding more clusters beyond 4 gives diminishing returns in reducing inertia.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []
K_range_sil = range(2, 11)

for k in K_range_sil:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

print('Silhouette scores computed for K = 2 to 10.')

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(K_range_sil, silhouette_scores, marker='o', color='darkorange', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score — Choosing K')
plt.xticks(K_range_sil)
plt.tight_layout()
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
score_table = pd.DataFrame({
    'K': list(K_range_sil),
    'Silhouette Score': [round(s, 6) for s in silhouette_scores]
})
score_table

**Output Interpretation**

A higher silhouette score means the clusters are more clearly separated. From this table, **K = 4** achieves a good balance — a relatively high silhouette score that is consistent with the elbow at K = 4. While higher K values may score slightly higher, they produce many small clusters that are harder to interpret from a business perspective.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
# K = 4 chosen based on the elbow curve and silhouette analysis
final_k = 4

final_model = KMeans(n_clusters=final_k, random_state=42, n_init=10)
final_labels = final_model.fit_predict(X_scaled)

print(f'Final model trained with K = {final_k}')

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = final_labels

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean().round(2)
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
counts = df['Cluster'].value_counts().sort_index()
print('Customers per cluster:')
print(counts)

counts.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(7, 4))
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.title('Customer Count per Cluster')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = final_labels

print(f'Explained variance by PC1: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'Explained variance by PC2: {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total explained variance : {pca.explained_variance_ratio_.sum()*100:.1f}%')

plt.figure(figsize=(9, 6))
colors = ['steelblue', 'tomato', 'green', 'orange']
for cluster_id in sorted(pca_df['Cluster'].unique()):
    subset = pca_df[pca_df['Cluster'] == cluster_id]
    plt.scatter(subset['PC1'], subset['PC2'],
                label=f'Cluster {cluster_id}',
                alpha=0.5, s=15,
                color=colors[cluster_id % len(colors)])

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Customer Segments — PCA Visualization (2D)')
plt.legend()
plt.tight_layout()
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions. Despite this, we can see that the clusters have distinct regions, confirming that K-Means has found meaningful separations in the original high-dimensional space.

## Final Questions

Answer the following questions:

**1. Why is this an unsupervised learning problem?**

This is an unsupervised learning problem because the dataset does not contain a target label or a pre-defined category for each customer. There is no column that says which segment a customer belongs to. Instead, we use K-Means to discover hidden patterns and group customers based on their behavior — without any supervision or prior knowledge of what the groups should look like.

**2. Why did we remove the `CUST_ID` column?**

`CUST_ID` is a unique identifier for each customer. It carries no information about customer behavior — it is just a label to tell customers apart. If we include it in clustering, K-Means would treat it as a numerical feature and try to cluster based on ID numbers, which is meaningless. Removing it ensures the algorithm only uses behaviorally relevant features.

**3. Which columns had missing values?**

Two columns had missing values:
- `CREDIT_LIMIT`: 1 missing value
- `MINIMUM_PAYMENTS`: 313 missing values

**4. How did you handle the missing values?**

We used **mean imputation** — replacing each missing value with the mean of that column. This approach is appropriate because both columns are continuous numerical features. The mean preserves the overall distribution of the data and avoids introducing bias by simply dropping rows, which could remove a significant portion of the dataset (especially for `MINIMUM_PAYMENTS` with 313 missing entries).

**5. Why is scaling important before applying K-Means?**

K-Means groups points based on **Euclidean distance**. If features have very different ranges — for example `BALANCE` ranges from 0 to ~19,000 while `PURCHASES_FREQUENCY` ranges from 0 to 1 — the large-scale features will completely dominate the distance calculations. Features with small ranges will have almost no influence on the clustering result. `StandardScaler` transforms each feature to have a mean of 0 and a standard deviation of 1, giving every feature equal influence in the distance computation.

**6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.**

We chose **K = 4**.

- **Elbow method:** The inertia curve shows a noticeable elbow around K = 4. After that point, adding more clusters gives diminishing reductions in inertia.
- **Silhouette score:** K = 4 achieves a solid silhouette score, indicating well-separated clusters. Although higher K values may score slightly higher, they produce too many small clusters that are difficult to interpret and act upon in a business context.
- **Business interpretation:** Four segments is a practical number for a credit card company to design distinct marketing strategies for each group.

**7. Based on the cluster summary table, describe each customer segment in your own words.**

Based on the cluster mean values:

- **Cluster 0 — Low-Activity Customers:** Low balance, low purchases, low cash advance. These customers barely use their credit card. They may be inactive or occasional users.
- **Cluster 1 — High-Balance Cash Advance Users:** High balance and high cash advance with low purchases. These customers rely on cash withdrawals rather than purchases and may carry debt. They represent a financial risk.
- **Cluster 2 — Active Purchasers:** High purchases (both one-off and installments), moderate balance, good payment behavior. These are engaged customers who actively shop with their card.
- **Cluster 3 — Transactors / Full Payers:** High credit limit, high payments, high PRC_FULL_PAYMENT. These customers pay their balance in full regularly — they are low-risk, high-value customers.

**8. Which cluster may represent high-value customers?**

**Cluster 3 (Transactors / Full Payers)** likely represents high-value customers. They have a high credit limit, make large payments, and tend to pay off their full balance regularly (`PRC_FULL_PAYMENT` is high). These customers are low risk for the bank and high in transaction volume, generating interchange fee revenue without defaulting on their balance.

**9. Which cluster may represent customers who rely more on cash advance?**

**Cluster 1 (High-Balance Cash Advance Users)** relies most heavily on cash advance. Their `CASH_ADVANCE`, `CASH_ADVANCE_FREQUENCY`, and `CASH_ADVANCE_TRX` values are significantly higher than other clusters. These customers withdraw cash from their credit card regularly, which typically carries higher interest rates and fees — indicating financial stress or a preference for liquidity over credit purchases.

**10. How can a company use these clusters for marketing strategy?**

| Cluster | Segment | Strategy |
|---|---|---|
| 0 | Low-Activity | Re-engagement campaigns, bonus rewards for first purchase, special introductory offers |
| 1 | Cash Advance Users | Offer balance transfer plans, debt consolidation products, financial counseling programs |
| 2 | Active Purchasers | Loyalty programs, cashback rewards on purchases, partner merchant deals |
| 3 | Full Payers | Premium card upgrades, travel rewards, exclusive perks to retain high-value customers |

Segmenting customers allows the company to move from a one-size-fits-all approach to **personalised marketing**, improving customer satisfaction, reducing churn, and increasing revenue.